### Mathematical Intuition

1. Random weights can make the signal grow or shrink layer-by-layer.
   - If variance keeps growing → exploding gradients.
   - If variance keeps shrinking → vanishing gradients.

2. Goal: keep **variance of activations the same** across every layer:
   $$
   \text{var}(a_{\text{out}}) = \text{var}(a_{\text{in}})
   $$

3. One-layer approximation (linear units, zero-mean data & weights):
   $$
   \text{var}(a_{\text{out}}) \approx n_{\text{in}} \cdot \text{var}(W) \cdot \text{var}(a_{\text{in}})
   $$
   where $n_{\text{in}}$ = number of fan-in weights.

4. Set $\text{var}(a_{\text{out}}) = \text{var}(a_{\text{in}})$ and solve for $\text{var}(W)$:
   $$
   \text{var}(W) = \frac{1}{n_{\text{in}}}
   $$

5. Standard Gaussian weights start with $\text{var}(W) = 1$.  
   Scale them by $c = \frac{1}{\sqrt{n_{\text{in}}}}$.  
   Variance after scaling:  
   $$
   \text{var}(c \cdot W) = c^{2} \cdot 1 = \frac{1}{n_{\text{in}}}
   $$  
   exactly what we need.

6. Result: activations keep stable variance, gradients neither explode nor vanish.

### Xavier/Glorot initialization
Xavier Initialization is particularly well-suited for activation functions like sigmoid and hyperbolic tangent (tanh), where the activation functions are sensitive to the scale of the inputs.

$$ \text{Normal variant: } \mathcal{N}\left(0, \frac{2}{\text{fan}_{\text{in}} + \text{fan}_{\text{out}}}\right) $$
<br>

$$ \text{Uniform variant: } \mathcal{U}\left(-\sqrt{\frac{6}{\text{fan}_{\text{in}} + \text{fan}_{\text{out}}}}, \sqrt{\frac{6}{\text{fan}_{\text{in}} + \text{fan}_{\text{out}}}}\right) $$

### He initialization
He Initialization is particularly effective for neural networks that use ReLU or its variants as activation functions.

$$ \text{Normal variant: } \mathcal{N}\left(0, \frac{1}{\text{fan\_mode}}\right) $$
<br>

$$ \text{Uniform variant: } \mathcal{U}\left(-\sqrt{\frac{3}{\text{fan\_mode}}}, \sqrt{\frac{3}{\text{fan\_mode}}}\right) $$

In [13]:
# Importing necessary libraries
import pandas as pd
import torch
from torch import nn

# Device
from typing import Literal
device: Literal['cpu', 'cuda'] = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
# Loading Dataset
df: pd.DataFrame = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
# Splitting the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [9]:
# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
# Data loading class
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, features, labels) -> None:
        super().__init__()
        self.features: torch.Tensor = torch.tensor(features, dtype = torch.float32)
        self.labels: torch.Tensor = torch.tensor(labels, dtype = torch.long) # for loss function, integer labels required.

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [12]:
# Creating an object
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Data Loaders
train_loader: DataLoader[torch.Tensor] = DataLoader(dataset = train_dataset, batch_size = 32, shuffle = True)
test_loader: DataLoader[torch.Tensor] = DataLoader(dataset = test_dataset, batch_size = 32, shuffle = True)

In [14]:
import torch.nn.init as init # Used to initialize the parameters

class CustomModel(nn.Module):
    def __init__(self, num_features):
        super(CustomModel, self).__init__()

        # Building model
        self.network = nn.Sequential(
            # For ReLU and its variants - He Initialization
            nn.Linear(in_features = num_features, out_features = 128),
            nn.LeakyReLU(negative_slope = 0.01),

            nn.Linear(in_features = 128, out_features = 64),
            nn.ELU(alpha = 1),

            nn.Linear(in_features = 64, out_features = 32),
            nn.PReLU(num_parameters = 1, init = 0.25),

            # For Tanh and sigmoid - Glorat Initialization
            nn.Linear(in_features = 32, out_features = 16),
            nn.Tanh(),

            nn.Linear(in_features = 16, out_features = 10),
            nn.Sigmoid()
        )

        # Initializing weights
        self.initialize_weights()

    def initialize_weights(self):
        # Find the layer in the sequential network to see what follows
        for i, module in enumerate(self.network):

            if i + 1 < len(self.network) and isinstance(module, nn.Linear):
                next_layer = self.network[i + 1] # To check the activation ahead of current hidden layer

                if isinstance(next_layer, (nn.ReLU, nn.LeakyReLU, nn.ELU, nn.PReLU)):
                    init.kaiming_normal_(tensor=module.weight, nonlinearity='leaky_relu') # mode: either 'fan_in' (default) or 'fan_out'. Choosing 'fan_in' preserves the magnitude of the variance of the weights in the forward pass. Choosing 'fan_out' preserves the magnitudes in the backwards pass.
                elif isinstance(next_layer, nn.Sigmoid):
                    gain = nn.init.calculate_gain(nonlinearity='sigmoid')
                    init.xavier_normal_(module.weight, gain=gain) # gain: an optional scaling factor
                
                # Constant Initialization - The type of parameters must be torch.nn.Parameter
                """
                    module.weight = torch.nn.Parameter(torch.zeros_like(module.weight, requires_grad=True))
                """

    def forward(self, X):
        return self.network(X)

In [15]:
# Model Object - Weights before optimization
model: CustomModel = CustomModel(num_features = X_train.shape[1]).to(device)

for i, module in enumerate(model.network):
    if isinstance(module, nn.Linear):
        print(module, module.weight, sep = '\n', end = '\n' + '_' * 50 + '\n')

Linear(in_features=30, out_features=128, bias=True)
Parameter containing:
tensor([[-0.3401,  0.2692, -0.3225,  ..., -0.3300, -0.1883,  0.1683],
        [-0.2217, -0.4107, -0.3990,  ..., -0.4660, -0.0361, -0.1069],
        [ 0.0319,  0.1348,  0.4009,  ..., -0.3877, -0.3363,  0.1555],
        ...,
        [ 0.5406,  0.2445, -0.0213,  ..., -0.0786,  0.0941, -0.3911],
        [-0.4530,  0.2872, -0.3148,  ..., -0.1420, -0.0854,  0.1926],
        [ 0.0607, -0.0748,  0.2589,  ..., -0.0348,  0.2820,  0.3877]],
       device='cuda:0', requires_grad=True)
__________________________________________________
Linear(in_features=128, out_features=64, bias=True)
Parameter containing:
tensor([[ 0.0708, -0.0100, -0.1747,  ...,  0.0283,  0.1712, -0.0294],
        [ 0.0814,  0.0559,  0.0744,  ...,  0.0726, -0.0636, -0.0508],
        [ 0.0579, -0.0227,  0.0055,  ..., -0.0044, -0.0067,  0.0297],
        ...,
        [-0.0154, -0.0755,  0.2153,  ..., -0.1131,  0.0204,  0.0338],
        [ 0.0925,  0.1450, -0.

In [16]:
# Compiling Model
learning_rate = 0.01
epochs = 10

# Defining Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(params = model.parameters(), lr = learning_rate)

In [18]:
# Training Loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:
        # Forward pass
        y_pred = model(batch_features.to(device))

        # Calculate loss
        loss = criterion(y_pred, batch_labels.to(device))

        # Zero Gradients
        optimizer.zero_grad()

        # Backward pass
        loss.backward()

        # Update Weights
        optimizer.step()

        # Epoch loss
        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.8249652147293092
Epoch: 2 , Loss: 1.5844894647598267
Epoch: 3 , Loss: 1.5230586926142375
Epoch: 4 , Loss: 1.5057162761688232
Epoch: 5 , Loss: 1.4895021677017213
Epoch: 6 , Loss: 1.484217675526937
Epoch: 7 , Loss: 1.4879198312759399
Epoch: 8 , Loss: 1.481639575958252
Epoch: 9 , Loss: 1.4796836455663045
Epoch: 10 , Loss: 1.4780680020650228


In [19]:
# Values of parameters after optimization
for i, module in enumerate(model.network):
    if isinstance(module, nn.Linear):
        print(module, module.weight, sep = '\n', end = '\n' + '_' * 50 + '\n')

Linear(in_features=30, out_features=128, bias=True)
Parameter containing:
tensor([[-0.3067,  0.4154, -0.2878,  ..., -0.5098, -0.3659,  0.0901],
        [-0.3877, -0.3564, -0.5818,  ..., -0.4533,  0.0199, -0.0016],
        [-0.0664,  0.0300,  0.3023,  ..., -0.3349, -0.3944,  0.1950],
        ...,
        [ 0.5004,  0.0589, -0.0608,  ..., -0.0986,  0.1853, -0.3305],
        [-0.3765,  0.3386, -0.2426,  ..., -0.1312,  0.0187,  0.1606],
        [ 0.1350, -0.2018,  0.3499,  ..., -0.0241,  0.3689,  0.3986]],
       device='cuda:0', requires_grad=True)
__________________________________________________
Linear(in_features=128, out_features=64, bias=True)
Parameter containing:
tensor([[ 0.0092, -0.0797, -0.2810,  ...,  0.0774,  0.1044,  0.0762],
        [ 0.1293,  0.0689,  0.1082,  ...,  0.1206, -0.0741, -0.0034],
        [ 0.1032,  0.0522,  0.0608,  ...,  0.0643,  0.1124,  0.1374],
        ...,
        [ 0.0325, -0.0381,  0.2901,  ..., -0.0294,  0.0562,  0.1179],
        [ 0.0445,  0.1889, -0.

In [20]:
model.eval()

CustomModel(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ELU(alpha=1)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): PReLU(num_parameters=1)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): Tanh()
    (8): Linear(in_features=16, out_features=10, bias=True)
    (9): Sigmoid()
  )
)

In [21]:
# Evaluating on Test Data - Demo
from torch import tensor
from torchmetrics.classification import MulticlassAccuracy

target = tensor([2, 1, 0, 0])
preds = tensor([2, 1, 0, 1])

metric = MulticlassAccuracy(num_classes=3)
metric(preds, target)

tensor(0.8333)

In [22]:
mca = MulticlassAccuracy(num_classes=3, average=None)
mca(preds, target)

tensor([0.5000, 1.0000, 1.0000])

In [27]:
# Evaluating on Test Data
accuracy = []
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features.to(device))
        _, predicted = torch.max(y_pred, 1)
        metric = MulticlassAccuracy(num_classes = 3, average=None).to(device)
        accuracy.append(metric(predicted, batch_labels.to(device)))
print(sum(accuracy) / len(accuracy))

tensor([0.9881, 0.9287, 0.0000], device='cuda:0')


### What Kaiming/He Initialization Does

#### The Core Goal

Neural networks multiply inputs by weights, layer after layer. If weights are too big, numbers explode. If too small, they vanish to zero. Good initialization keeps the **scale roughly 1:1** — what goes in has similar magnitude to what comes out.

---

#### Forward Pass: Preserving Activations

For a linear layer with ReLU activation, the forward pass is:

$$y = \text{ReLU}(x \cdot W^T)$$

Kaiming init scales weights so that **the variance of outputs ≈ variance of inputs**:

$$\text{Var}(W) \approx \frac{2}{\text{fan\_mode}}$$

The factor of 2 accounts for ReLU "killing" half the values (setting negatives to zero) .

---

#### Backward Pass: Preserving Gradients

During backpropagation, gradients flow backward through the layer like this :

$$\delta_{l-1} = W_l^T \cdot \delta_l$$

Where:
- $\delta_l = \frac{\partial \text{Loss}}{\partial y_l}$ = gradient flowing backward from the next layer
- $\delta_{l-1} = \frac{\partial \text{Loss}}{\partial x_l}$ = gradient with respect to this layer's input

The variance of the backward-flowing gradient is:

$$\text{Var}(\delta_{l-1}) \approx \text{fan\_out} \times \text{Var}(W) \times \text{Var}(\delta_l)$$

To perfectly preserve gradient variance, you'd need:

$$\text{Var}(W) \approx \frac{1}{\text{fan\_out}} \quad \text{(fan\_out mode)}$$

---

#### The Trade-Off: You Can't Optimize Both Perfectly

| | What you'd need | What Kaiming uses (default) |
|---|---|---|
| **Forward pass** | $\text{Var}(W) \approx \frac{2}{\text{fan\_mode}}$ | $\text{Var}(W) \approx \frac{2}{\text{fan\_mode}}$ ✅ |
| **Backward pass** | $\text{Var}(W) \approx \frac{1}{\text{fan\_out}}$ | Same as above (approximate) |

**You can't perfectly optimize for both simultaneously unless $\text{fan\_mode} = \text{fan\_out}$.**

Kaiming's default `fan_in` mode prioritizes **forward pass stability**. This is a deliberate trade-off — forward stability is usually more important for ReLU networks, and it happens to keep gradients healthy enough in practice . The math works out that if you preserve one, you (approximately) preserve the other.

---

#### Why the Transpose Matters

PyTorch's `Linear` layer stores weights as $W$ with shape $[\text{fan\_out}, \text{fan\_mode}]$, but computes:

$$\text{output} = x \cdot W^T$$

So `nn.init.kaiming_normal_(W, ...)` assumes $W[0]$ is a row of `fan_in` elements. If your custom weight matrix is $[\text{fan\_mode}, \text{fan\_out}]$ (the opposite), pass `W.T` to the init function so it correctly identifies which dimension is `fan_in` vs `fan_out` .

---

#### Simple Summary

| | |
|---|---|
| **Goal** | Keep numbers from exploding/vanishing through many layers |
| **How** | Scale weights by $\sim 1/\sqrt{\text{fan\_mode}}$ |
| **Preserves forward** | Variance of activations (data flowing through) |
| **Preserves backward** | Variance of $\frac{\partial \text{Loss}}{\partial x}$ (gradients w.r.t. layer inputs) |
| **The catch** | Perfect preservation of both requires $\text{fan\_mode} = \text{fan\_out}$; Kaiming defaults to forward stability |

The "preserve gradients in backward pass" means: **the gradient of the loss with respect to each layer's input doesn't explode or vanish as it propagates backward**, so early layers get gradients of healthy magnitude to learn from .